## Imports

In [40]:
import math
import os
import time
import requests
import pandas as pd
import json
import numpy as np
from pathlib import Path
from typing import Optional, Dict, List, Union
from unidecode import unidecode
import matplotlib.pyplot as plt

import pandas_gbq
from google.auth import default
from google.cloud import bigquery
from google.api_core.exceptions import NotFound

import warnings
warnings.filterwarnings("ignore")

In [41]:
from funcoes_escoragem import *

## Diretório

In [42]:
BASE_DIR = Path("data")
RAW_DIR = BASE_DIR / "raw"
TRUSTED_DIR = BASE_DIR / "trusted"
ANALYTICS_DIR = BASE_DIR / "analytics"

for path in [RAW_DIR, TRUSTED_DIR, ANALYTICS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

## Google BigQuery

In [43]:
project_id = 'loft-dl-fintech'

# CredPago - Consulta Realizada**

In [44]:
query_credpago = f"""
WITH consulta_realizada AS (
    SELECT
        CAST(REGEXP_REPLACE(documento, r'[^0-9]', '') AS INT64) AS CPF_CNPJ,
        id_externo AS contract_id,

        MIN(DATE(data)) OVER (
            PARTITION BY id_externo
        ) AS requested_at,

        MAX(DATE(data)) OVER (
            PARTITION BY id_externo
        ) AS data_ultima_consulta,
        ARRAY_LENGTH(JSON_EXTRACT_ARRAY(CR.json_retornado, '$.pessoas')) AS qtd_proponentes,
        CR.*
    FROM `loft-dl-fintech.bronze_credpago_sortinghat.consulta_realizada` CR
    WHERE DATE(data) >= DATE_SUB(CURRENT_DATE(), INTERVAL 4 WEEK)
    AND DATE(data) < CURRENT_DATE()
)

SELECT *
FROM consulta_realizada
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY contract_id
    ORDER BY 
        CASE WHEN model = 'BLEND_4' THEN 1 ELSE 2 END ASC,
        data DESC
) = 1
"""

credpago_df = pd.read_gbq(query_credpago, project_id=project_id)
credpago_df

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,renda_considerada,model,id_externo,json_retornado,...,id_funcionalidade,_sdc_table_version,request,_sdc_received_at,_sdc_sequence,documento,rating,_sdc_batched_at,data,result
0,11671492978,4118927,2026-06-16,2026-07-06,1,NI,1438.500000000,BLEND3_3,4118927,"{""pessoas"":[{""nome"":""MARIA EDUARDA MENGUE DA C...",...,34,1778785248777,"{""valor_aluguel"":750,""matchmaking_on"":false,""p...",2026-07-07 08:00:35+00:00,1783411230009168727,11671492978,C,2026-07-07 08:02:56.154000+00:00,2026-07-06 15:58:07+00:00,APROVAR
1,3949480099,4198070,2026-06-17,2026-06-17,1,B,11850.500000000,BLEND_REGRESSAO_2026,4198070,"{""pessoas"":[{""nome"":""MARIANA GROSS DE MOURA"",""...",...,32,1778785248777,"{""valor_aluguel"":""5500"",""imobiliaria_id"":583,""...",2026-06-18 08:00:46+00:00,1781769642872972222,03949480099,B,2026-06-18 08:04:35.239000+00:00,2026-06-17 16:42:01+00:00,APROVAR
2,78534607087,4208123,2026-06-19,2026-06-22,1,B,13700.000000000,BLEND3_3,4208123,"{""pessoas"":[{""nome"":""SEURA ELISABTHE DA SILVA""...",...,34,1778785248777,"{""valor_aluguel"":""2000"",""imobiliaria_id"":1752,...",2026-06-23 08:00:35+00:00,1782201632912360978,78534607087,C,2026-06-23 08:12:37.597000+00:00,2026-06-22 16:58:05+00:00,APROVAR
3,61425336329,4208366,2026-06-20,2026-06-20,1,C,1370.000000000,BLEND_REGRESSAO_2026,4208366,"{""pessoas"":[{""nome"":""ROBERTA ARAUJO DE SOUSA"",...",...,34,1778785248777,"{""valor_aluguel"":""1500"",""imobiliaria_id"":923,""...",2026-06-20 18:00:25+00:00,1781978420674086670,61425336329,C,2026-06-20 18:05:09.820000+00:00,2026-06-20 08:08:59+00:00,APROVAR
4,55578207871,4216609,2026-06-16,2026-06-16,1,B,1644.000000000,BLEND3_3,4216609,"{""pessoas"":[{""nome"":""BRUNO HENRIQUE SPROVIERI ...",...,33,1778785248777,"{""valor_aluguel"":""1100"",""imobiliaria_id"":12354...",2026-06-16 18:00:37+00:00,1781632832875088651,55578207871,E,2026-06-16 18:03:52.897000+00:00,2026-06-16 10:37:04+00:00,APROVAR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112147,35687814841,4367526,2026-07-13,2026-07-13,1,NI,15481.000000000,BLEND3_3,4367526,"{""pessoas"":[{""nome"":""TATIANA ALVES DA SILVA"",""...",...,31,1778785248777,"{""valor_aluguel"":5002,""matchmaking_on"":false,""...",2026-07-14 08:00:25+00:00,1784016023798958260,35687814841,A,2026-07-14 08:03:49.764000+00:00,2026-07-13 18:48:01+00:00,APROVAR
112148,3754766104,4367561,2026-07-13,2026-07-13,1,C,1644.000000000,BLEND3_3,4367561,"{""pessoas"":[{""nome"":""DAYRA IVETTE SAURI VAZQUE...",...,34,1778785248777,"{""valor_aluguel"":1400,""matchmaking_on"":false,""...",2026-07-14 08:00:25+00:00,1784016023864143088,03754766104,C,2026-07-14 08:03:49.788000+00:00,2026-07-13 19:10:07+00:00,APROVAR
112149,41343421802,4367602,2026-07-13,2026-07-13,1,NI,4521.000000000,BLEND3_3,4367602,"{""pessoas"":[{""nome"":""GUILHERME TEIXEIRA"",""docu...",...,32,1778785248777,"{""valor_aluguel"":2000,""matchmaking_on"":false,""...",2026-07-14 08:00:25+00:00,1784016023940284399,41343421802,B,2026-07-14 08:03:49.814000+00:00,2026-07-13 19:40:35+00:00,APROVAR
112150,12320399186,4367677,2026-07-13,2026-07-13,1,NI,1370.000000000,BLEND3_3,4367677,"{""pessoas"":[{""nome"":""VANIA DANILO SAIDE"",""docu...",...,33,1778785248777,"{""valor_aluguel"":1250,""matchmaking_on"":false,""...",2026-07-14 08:00:25+00:00,1784016024072854374,12320399186,E,2026-07-14 08:03:49.870000+00:00,2026-07-13 21:14:26+00:00,APROVAR


In [45]:
mask = credpago_df['contract_id'].astype(str).str.match(r'^\d+$')
credpago_df = credpago_df[mask].copy()
credpago_df['contract_id'] = credpago_df['contract_id'].astype(int)

In [46]:
credpago_df['model'].value_counts(dropna=False)

model
BLEND3_3                95420
BLEND_REGRESSAO_2026     6006
HVA3                     4394
BLEND_4                  4144
BVS_CUSTOM               1360
HVA4                      695
BVS_CUSTOM_V2             102
HFT1                       17
                           14
Name: count, dtype: int64

## Multiproponente

In [47]:
credpago_df['qtd_proponentes'].value_counts(dropna=False)

qtd_proponentes
1    107823
2      3976
3       319
4        31
8         1
6         1
5         1
Name: count, dtype: Int64

In [48]:
credpago_df['qtd_proponentes'].value_counts(normalize=True, dropna=False)

qtd_proponentes
1    0.961401
2    0.035452
3    0.002844
4    0.000276
8    0.000009
6    0.000009
5    0.000009
Name: proportion, dtype: Float64

## Quebrar JSON - Retornado

In [49]:
def unwrap_payload(obj):
    """Supports old format (message wrapper) and new format (root payload)."""
    if not obj:
        return None
    if isinstance(obj, dict) and "message" in obj:
        return obj["message"]
    return obj

def get_pessoas(obj):
    payload = unwrap_payload(obj)
    return (payload or {}).get("pessoas") or []

In [50]:
parsed = credpago_df["json_retornado"].apply(
    lambda x: json.loads(x) if pd.notna(x) and x else None
)

valid_parsed = parsed.dropna()
payload_parsed = valid_parsed.apply(unwrap_payload)

# json_normalize resets the index; preserve credpago_df labels for the join below
contrato_df = pd.json_normalize(payload_parsed.tolist(), sep="_")
contrato_df.index = payload_parsed.index
contrato_df = contrato_df.add_prefix("message_")  # keeps message_decisao, message_scores_*, etc.

pessoa_records = payload_parsed.apply(lambda x: (get_pessoas(x) or [{}])[0])
pessoa_df = pd.json_normalize(pessoa_records.tolist(), sep="_")
pessoa_df.index = payload_parsed.index
pessoa_df = pessoa_df.add_prefix("pessoa_")

In [51]:
valid = parsed.notna()
base_idx = credpago_df.loc[valid].index

pendencias = []
for idx, row in parsed[valid].items():
    for p in get_pessoas(row):
        if not isinstance(p, dict):
            continue

        serasa = p.get("anotacoesFinanceirasSerasa") or {}
        pend_list = serasa.get("PendenciasPagamentoPF") or []

        for pend in pend_list:
            if isinstance(pend, dict):
                pendencias.append({"idx": idx, **pend})

pendencias_df = pd.DataFrame(pendencias)

if not pendencias_df.empty:
    pendencias_agg = (
        pendencias_df
        .groupby("idx", as_index=False)
        .agg(
            qtde_pefin=("Valor", "count"),
            valor_pefin_total=("Valor", lambda s: pd.to_numeric(s, errors="coerce").sum()),
            modalidades_pefin=("Modalidade", lambda s: ",".join(sorted(set(s.dropna().astype(str))))),
        )
    )
else:
    pendencias_agg = pd.DataFrame(columns=["idx", "qtde_pefin", "valor_pefin_total", "modalidades_pefin"])

## Quebrar JSON - Request


In [52]:
request_parsed = credpago_df["request"].apply(
    lambda x: json.loads(x) if pd.notna(x) and x else None
)

request_valid = request_parsed.dropna()
request_df = pd.json_normalize(request_valid.tolist(), sep="_")
request_df.index = request_valid.index
request_df = request_df.add_prefix("request_")

# Unify old (snake_case) and new (camelCase) request schemas
REQUEST_FIELD_ALIASES = {
    "request_valorAluguel": ["request_valor_aluguel"],
    "request_imobiliariaId": ["request_imobiliaria_id"],
    "request_idExterno": ["request_id_externo"],
    "request_imovelTipo": ["request_imovel_tipo"],
    "request_principalDocument": ["request_pessoa_principal_documento"],
}

for target, sources in REQUEST_FIELD_ALIASES.items():
    existing_sources = [c for c in sources if c in request_df.columns]
    if not existing_sources and target not in request_df.columns:
        continue
    if target not in request_df.columns:
        request_df[target] = pd.NA
    for source in existing_sources:
        if target == "request_valorAluguel":
            request_df[target] = request_df[target].combine_first(
                pd.to_numeric(request_df[source], errors="coerce")
            )
        else:
            request_df[target] = request_df[target].combine_first(request_df[source])
        request_df = request_df.drop(columns=[source])


## Join Json's

In [53]:
EXPANDED_PREFIXES = ("message_", "pessoa_", "request_")
expanded_cols = [c for c in credpago_df.columns if c.startswith(EXPANDED_PREFIXES)]

base_df = credpago_df.loc[valid].drop(columns=expanded_cols, errors="ignore")

credpago_expandido = (
    base_df
    .join(contrato_df, how="left")
    .join(pessoa_df, how="left")
    .join(request_df, how="left")
    .reset_index(names="idx")
    .merge(pendencias_agg, on="idx", how="left")
    .drop(columns="idx")
)


In [54]:
cols_drop = [
    # ingestão
    "_sdc_table_version", "_sdc_received_at", "_sdc_sequence", "_sdc_batched_at",

    # json bruto / containers
    "json_retornado",
    "request",
    "message_pessoas", "message_scores", "message_ratings",
    "pessoa_scores", "pessoa_ratings", "message_blend3Variables",
    "request_priorDecisions", "request_dadosAusentes", "request_errosTecnicos",
    "request_outras_pessoas", "request_blend3Variables", "request_scores",

    # bronze redundante
    "score_bvs",
    "renda_considerada",
    "decisao_bureau",
    "limite_mensal",

    # prováveis duplicatas
    "id_externo",
    "data",
    "request_month",

    # operacional / baixo valor (old format only — safe to keep)
    "success", "message_manual", "message_node",
    "message_reaproveitamentoConsultaMotor", "message_savingBureauPowerCurve",
    "message_logs", "message_partnerIds",
    "message_errosTecnicos", "message_dadosAusentes",
    "pessoa_errosTecnicos", "pessoa_dadosAusentes",
    "message_rentGuaranteeConstraints_rent_coverage_multiplier",
    "message_rentGuaranteeConstraints_exit_cost_multiplier",
    "message_rentGuaranteeConstraints_commission_percent",
    "message_rentGuaranteeConstraints_version",
]

credpago_clean = credpago_expandido.drop(columns=[c for c in cols_drop if c in credpago_expandido.columns])


In [55]:
credpago_clean[
    (pd.to_datetime(credpago_clean["requested_at"]).dt.normalize() == "2026-07-03")
    & (credpago_clean["message_decisao"] == "BLEND_4")
][["message_blend3Variables_realEstatePc4HistoryOver100Contracts", "message_blend3Variables_cityPc4HistoryOver100Contracts"]]

,message_blend3Variables_realEstatePc4HistoryOver100Contracts,message_blend3Variables_cityPc4HistoryOver100Contracts
763,0.000000,0.089384
778,0.000000,0.107708
783,0.000000,0.054962
1919,0.000000,0.138790
4193,0.000000,0.130864
...,...,...
110729,0.000000,0.072148
110736,0.000000,0.000000
111804,0.000000,0.123858
111806,0.000000,0.000000


## Escoragem Blend4

In [56]:
credpago_clean_rep = credpago_clean.replace({
    "request_imovelTipo": {"RESIDENCIAL": 1, "COMERCIAL": 0}, 
    "message_blend3Variables_hasPreviousContracts": {True: 1, False: 0},
    "message_blend3Variables_hadOverdueInvoiceInPreviousContracts": {True: 1, False: 0}
}
)

In [57]:
rename_dict = {
    "message_scores_BVS_CUSTOM_V2": "score_proposto__bvs",#
    "message_scores_HFT1": "SERASA_CHSV5",
    "pessoa_idade": "age",
    "request_imovelTipo": "property_type",
    "message_qtdeRestricoesContrato": "qtde_restricoes__consulta_realizada",
    "request_valorAluguel": "rental_value",
    "message_rendaConsideradaContrato": "income",
    "message_blend3Variables_realEstatePc4HistoryOver100Contracts": "agency_pc4_mais_100_contratos__pc_categorias",
    "message_blend3Variables_cityPc4HistoryOver100Contracts": "city_pc4_mais_100_contratos__pc_categorias",
    "message_blend3Variables_hasPreviousContracts": "flag_tem__contratos_anteriores",
    "message_blend3Variables_hadOverdueInvoiceInPreviousContracts": "flag_teve_boleto_atrasado__contratos_anteriores",
}

In [58]:
vars_blend4 = ['score_proposto__bvs', 'SERASA_CHSV5', 'age', 'property_type', 'qtde_restricoes__consulta_realizada', 'rental_value', 'income', 'agency_pc4_mais_100_contratos__pc_categorias', 'city_pc4_mais_100_contratos__pc_categorias', 'flag_tem__contratos_anteriores', 'flag_teve_boleto_atrasado__contratos_anteriores']

id_vars = ['contract_id', 'requested_at', 'CPF_CNPJ', 'message_decisao', 'message_blendRegressaoPredict']

In [59]:
df_predict = (credpago_clean_rep.copy()).rename(columns=rename_dict)
df_predict["REGRA_BLEND_4"] = np.where(
    df_predict["score_proposto__bvs"] < 334,
    "E_BVS",
    "BLEND4",
)

df_predict["SCORE_BVS"] = df_predict["score_proposto__bvs"]
df_predict["SCORE_SERASA"] = df_predict["SERASA_CHSV5"]
df_predict["RENDA"] = df_predict["income"]

df_predict.head()

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,model,id_consulta,id_funcionalidade,documento,...,request_blend3Variables_hadOverdueInvoiceInPreviousContracts,request_blend3Variables_cityPc4HistoryOver100Contracts,request_blend3Variables_realEstatePc4HistoryOver100Contracts,qtde_pefin,valor_pefin_total,modalidades_pefin,REGRA_BLEND_4,SCORE_BVS,SCORE_SERASA,RENDA
0,11671492978,4118927,2026-06-16,2026-07-06,1,NI,BLEND3_3,6053612,34,11671492978,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,NaN,NaN,1438.5
1,3949480099,4198070,2026-06-17,2026-06-17,1,B,BLEND_REGRESSAO_2026,5964289,32,03949480099,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,NaN,NaN,11850.5
2,78534607087,4208123,2026-06-19,2026-06-22,1,B,BLEND3_3,5986112,34,78534607087,...,NaN,NaN,NaN,3.0,1276.0,"CRED CARTAO,OUTRAS OPER",BLEND4,NaN,NaN,13700.0
3,61425336329,4208366,2026-06-20,2026-06-20,1,C,BLEND_REGRESSAO_2026,5977913,34,61425336329,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,NaN,NaN,1370.0
4,55578207871,4216609,2026-06-16,2026-06-16,1,B,BLEND3_3,5952969,33,55578207871,...,NaN,NaN,NaN,1.0,156.0,CRED CARTAO,BLEND4,NaN,NaN,1644.0


In [60]:
df_predict.groupby("REGRA_BLEND_4", dropna=False).size()

REGRA_BLEND_4
BLEND4    111730
E_BVS        422
dtype: int64

In [61]:
# df_predict.to_csv(ANALYTICS_DIR/"df_predict_blend4_pre.csv", index=False)

## Criacão das Variáveis

In [62]:
df_predict_vars = prepare_blend4_variables(df_predict)
df_predict_escorada = predict_blend4_1(df_predict_vars)

## Criar Rating Blend4

In [63]:
score = pd.to_numeric(df_predict_escorada["pred_blend4_1_to_score"], errors="coerce")

conditions = [
    score.between(480, 1000),
    score.between(443, 479),
    score.between(408, 442),
    score.between(343, 407),
    score.between(0, 342),
]

choices = ["A", "B", "C", "D", "E"]

df_predict_escorada["rating_manual_blend4"] = np.select(conditions, choices, default=None)

In [64]:
score = pd.to_numeric(df_predict_escorada["message_blendRegressaoPredict"], errors="coerce")

conditions = [
    score.between(480, 1000),
    score.between(443, 479),
    score.between(408, 442),
    score.between(343, 407),
    score.between(0, 342),
]

choices = ["A", "B", "C", "D", "E"]

df_predict_escorada["rating_json_blend4"] = np.select(conditions, choices, default=None)

## Salvar Base como se fosse Blend4

In [65]:
# df_predict_blend4 = df_predict_escorada[df_predict_escorada["message_decisao"] == "BLEND3_3"].copy()
# df_predict_blend4["message_decisao"] = df_predict_blend4["message_decisao"].replace("BLEND3_3", "BLEND4")

# cp_final_df = pd.concat([df_predict_escorada, df_predict_blend4])
cp_final_df = df_predict_escorada.copy()
cp_final_df

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,model,id_consulta,id_funcionalidade,documento,...,SERASA_CHSV5__normalized4_1,age__normalized4_1,qtde_restricoes__consulta_realizada__normalized4_1,income_commitment__normalized4_1,agency_pc4_mais_100_contratos__pc_categorias__normalized4_1,city_pc4_mais_100_contratos__pc_categorias__normalized4_1,pred_blend4_1,pred_blend4_1_to_score,rating_manual_blend4,rating_json_blend4
0,11671492978,4118927,2026-06-16,2026-07-06,1,NI,BLEND3_3,6053612,34,11671492978,...,0.0,-0.70,0.0,0.201963,0.000000,0.000000,0.495999,504.0,A,A
1,3949480099,4198070,2026-06-17,2026-06-17,1,B,BLEND_REGRESSAO_2026,5964289,32,03949480099,...,0.0,-0.45,0.0,0.075479,0.000000,0.000000,0.517687,482.0,A,E
2,78534607087,4208123,2026-06-19,2026-06-22,1,B,BLEND3_3,5986112,34,78534607087,...,0.0,0.85,3.0,-0.627237,-0.032295,-0.844994,0.523165,477.0,B,A
3,61425336329,4208366,2026-06-20,2026-06-20,1,C,BLEND_REGRESSAO_2026,5977913,34,61425336329,...,0.0,-0.40,0.0,1.468796,0.000000,0.000000,0.525219,475.0,B,E
4,55578207871,4216609,2026-06-16,2026-06-16,1,B,BLEND3_3,5952969,33,55578207871,...,0.0,-0.70,1.0,0.528269,0.000000,0.052119,0.533198,467.0,B,E
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112147,35687814841,4367526,2026-07-13,2026-07-13,1,NI,BLEND3_3,6090114,31,35687814841,...,0.0,0.15,2.0,-0.235996,0.000000,0.768522,0.547219,453.0,B,A
112148,3754766104,4367561,2026-07-13,2026-07-13,1,C,BLEND3_3,6090158,34,03754766104,...,0.0,-0.15,0.0,0.931352,0.000000,0.124206,0.537184,463.0,B,C
112149,41343421802,4367602,2026-07-13,2026-07-13,1,NI,BLEND3_3,6090204,32,41343421802,...,0.0,-0.60,2.0,0.027468,0.000000,-0.348689,0.514923,485.0,A,B
112150,12320399186,4367677,2026-07-13,2026-07-13,1,NI,BLEND3_3,6090304,33,12320399186,...,0.0,-0.65,0.0,1.065713,0.000000,0.640266,0.558596,441.0,C,E


In [66]:
cp_final_df.groupby("message_decisao", dropna=False).size()

message_decisao
                           14
BLEND3_3                95420
BLEND_4                  4144
BLEND_REGRESSAO_2026     6006
BVS_CUSTOM               1360
BVS_CUSTOM_V2             102
HFT1                       17
HVA3                     4394
HVA4                      695
dtype: int64

In [67]:
bvs = pd.to_numeric(cp_final_df["SCORE_BVS"], errors="coerce")
score = pd.to_numeric(cp_final_df["pred_blend4_1_to_score"], errors="coerce")

conditions = [
    bvs <= 334,                         # corte customizado BVS → E
    score.between(763, 1000),           # 763 – 1000
    score.between(704, 762),            # 704 – 762
    score.between(653, 703),            # 653 – 703
    score.between(607, 652),            # 607 – 652
    score.between(562, 606),            # 562 – 606
    score.between(520, 561),            # 520 – 561
    score.between(480, 519),            # 480 – 519
    score.between(443, 479),            # 443 – 479
    score.between(408, 442),            # 408 – 442
    score.between(375, 407),            # 375 – 407
    score.between(343, 374),            # 343 – 374
    score.between(307, 342),            # 307 – 342
    score.between(0, 306),              # 0 – 306
]

choices = [
    "E.E",      # override BVS ≤ 334
    "1.A+",
    "2.A",
    "3.A",
    "4.B+",
    "5.B+",
    "6.B",
    "7.B",
    "8.C",
    "9.D+",
    "10.D",
    "11.D",
    "E-1.E",
    "E-2.E",
]

cp_final_df["rating_cl_pol_blend4"] = np.select(conditions, choices, default=None)

## Salvar

In [68]:
cp_final_df.groupby("REGRA_BLEND_4", dropna=False).size()

REGRA_BLEND_4
BLEND4    111730
E_BVS        422
dtype: int64

In [69]:
cp_final_df.groupby("message_decisao", dropna=False).size()

message_decisao
                           14
BLEND3_3                95420
BLEND_4                  4144
BLEND_REGRESSAO_2026     6006
BVS_CUSTOM               1360
BVS_CUSTOM_V2             102
HFT1                       17
HVA3                     4394
HVA4                      695
dtype: int64

In [70]:
cp_final_df.groupby("model", dropna=False).size()

model
                           14
BLEND3_3                95420
BLEND_4                  4144
BLEND_REGRESSAO_2026     6006
BVS_CUSTOM               1360
BVS_CUSTOM_V2             102
HFT1                       17
HVA3                     4394
HVA4                      695
dtype: int64

In [71]:
cp_final_df.groupby("REGRA_BLEND_4", dropna=False).size()

REGRA_BLEND_4
BLEND4    111730
E_BVS        422
dtype: int64

In [72]:
cp_final_df.to_csv(ANALYTICS_DIR/"df_predict_blend4.csv", index=False)